# 模型数据准备与分析前检查

主目标：治疗前MRI和临床治疗信息是否提供增量预测信息。
次目标：在同一人群、预测时点及结局窗口下评价早期纵向信息。
死亡需要作为竞争事件处理；不能直接把随访不足或早期死亡归为12个月无事件。


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/rn_public').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from rn_public.bundle import restore_sources
print('Source restoration:', restore_sources(ROOT))
print('Project root:', ROOT.name)


Source restoration: {'restored_files': 0, 'status': 'all_source_checksums_verified'}
Project root: rn-mri-clinical-fusion


In [2]:
review = pd.read_csv(ROOT / 'data/processed/outcome_review.csv')
print(review.review_status.value_counts().to_string())
required = ['adjudicated_rn_event','reference_standard','evidence_source']
verified = review.review_status.isin(['verified_event','verified_event_free'])
complete = verified & review[required].notna().all(axis=1)
print('Outcome-reviewed lesion rows:', int(complete.sum()))
print('Review fields must be filled from traceable public source evidence; no values are invented.')


review_status
pending    160
Outcome-reviewed lesion rows: 0
Review fields must be filled from traceable public source evidence; no values are invented.


In [3]:
analysis_plan = pd.DataFrame([
    ['M0','Prespecified clinical and treatment predictors','Penalized competing-risk/survival framework'],
    ['M1','M0 plus baseline MRI features','Same patients, outcome horizon and validation splits as M0'],
    ['M2 (optional)','M1 plus event-preceding early MRI changes','Fixed landmark; compare only within the landmark cohort'],
], columns=['model','inputs','design'])
print(analysis_plan.to_string(index=False))
print('Validation: patient-level resampling; preprocessing and feature selection within training folds.')
print('Evaluation: discrimination, calibration, Brier score and uncertainty; no external validation claim without matching endpoints.')


        model                                         inputs                                                     design
           M0 Prespecified clinical and treatment predictors                Penalized competing-risk/survival framework
           M1                  M0 plus baseline MRI features Same patients, outcome horizon and validation splits as M0
M2 (optional)      M1 plus event-preceding early MRI changes    Fixed landmark; compare only within the landmark cohort
Validation: patient-level resampling; preprocessing and feature selection within training folds.
Evaluation: discrimination, calibration, Brier score and uncertainty; no external validation claim without matching endpoints.


In [4]:
readiness = {
    'outcomes_adjudicated': bool(complete.sum() > 0),
    'target_lesion_segmentation_qc_complete': False,
    'prediction_horizon_and_censoring_confirmed': False,
}
print(pd.Series(readiness).to_string())
if not all(readiness.values()):
    print('MODEL FITTING NOT STARTED: the current release is a data acquisition and curation project.')


outcomes_adjudicated                          False
target_lesion_segmentation_qc_complete        False
prediction_horizon_and_censoring_confirmed    False
MODEL FITTING NOT STARTED: the current release is a data acquisition and curation project.
